In [1]:
import os, kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
path = kagglehub.dataset_download("arianazmoudeh/airbnbopendata")
print(os.listdir(path))

data = pd.read_csv(os.path.join(path, "Airbnb_Open_Data.csv"), low_memory=False)
print(f"Entire Dataset successfully loaded : {data.shape}")
data.head()

100%|██████████| 10.5M/10.5M [00:01<00:00, 6.79MB/s]

Extracting files...


['Airbnb_Open_Data.csv']
Entire Dataset successfully loaded : (102599, 26)


,id,NAME,host id,host_identity_verified,host name,neighbourhood group,neighbourhood,lat,long,country,...,service fee,minimum nights,number of reviews,last review,reviews per month,review rate number,calculated host listings count,availability 365,house_rules,license
0,1001254,Clean & quiet apt home by the park,80014485718,unconfirmed,Madaline,Brooklyn,Kensington,40.64749,-73.97237,United States,...,$193,10.0,9.0,10/19/2021,0.21,4.0,6.0,286.0,Clean up and treat the home the way you'd like...,NaN
1,1002102,Skylit Midtown Castle,52335172823,verified,Jenna,Manhattan,Midtown,40.75362,-73.98377,United States,...,$28,30.0,45.0,5/21/2022,0.38,4.0,2.0,228.0,Pet friendly but please confirm with me if the...,NaN
2,1002403,THE VILLAGE OF HARLEM....NEW YORK !,78829239556,NaN,Elise,Manhattan,Harlem,40.80902,-73.94190,United States,...,$124,3.0,0.0,NaN,NaN,5.0,1.0,352.0,"I encourage you to use my kitchen, cooking and...",NaN
3,1002755,NaN,85098326012,unconfirmed,Garry,Brooklyn,Clinton Hill,40.68514,-73.95976,United States,...,$74,30.0,270.0,7/5/2019,4.64,4.0,1.0,322.0,NaN,NaN
4,1003689,Entire Apt: Spacious Studio/Loft by central park,92037596077,verified,Lyndon,Manhattan,East Harlem,40.79851,-73.94399,United States,...,$41,10.0,9.0,11/19/2018,0.10,3.0,1.0,289.0,"Please no smoking in the house, porch or on th...",NaN


In [3]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 102599 entries, 0 to 102598
Data columns (total 26 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   id                              102599 non-null  int64  
 1   NAME                            102349 non-null  object 
 2   host id                         102599 non-null  int64  
 3   host_identity_verified          102310 non-null  object 
 4   host name                       102193 non-null  object 
 5   neighbourhood group             102570 non-null  object 
 6   neighbourhood                   102583 non-null  object 
 7   lat                             102591 non-null  float64
 8   long                            102591 non-null  float64
 9   country                         102067 non-null  object 
 10  country code                    102468 non-null  object 
 11  instant_bookable                102494 non-null  object 
 12  cancellation_pol

In [4]:
#check for missing values in each column before we proceed to clean
missing_data = data.isna().sum().sort_values(ascending=False)
missing_data

,0
license,102597
house_rules,52131
last review,15893
reviews per month,15879
country,532
availability 365,448
minimum nights,409
host name,406
review rate number,326
calculated host listings count,319


In [5]:
#check for any duplicate row
data_duplicate = data.duplicated().sum()
data_duplicate

np.int64(541)

In [6]:
#drop duplicates
before = len(data)
data = data.drop_duplicates().reset_index(drop=True)
print(f"Before with {data_duplicate} duplicates: {before}, after removing duplicates: {len(data)}")

Before with 541 duplicates: 102599, after removing duplicates: 102058


In [7]:
#drop unnecessary columns - license, almost all entries are null; house rules - 50% null with free text unrelated to demand,
#country, country_code - only 1 value United States, so no information, review rate number - ratings on listings with zero reviews, logically impossible
#last review - impossible future dates

cols_to_drop = ['license', 'house_rules', 'country', 'country code', 'review rate number', 'last review']

data = data.drop(columns=cols_to_drop)
print(data.shape)


(102058, 20)


In [8]:
#columns now in our dataset
data.columns

Index(['id', 'NAME', 'host id', 'host_identity_verified', 'host name',
       'neighbourhood group', 'neighbourhood', 'lat', 'long',
       'instant_bookable', 'cancellation_policy', 'room type',
       'Construction year', 'price', 'service fee', 'minimum nights',
       'number of reviews', 'reviews per month',
       'calculated host listings count', 'availability 365'],
      dtype='object')

In [9]:
#lets clean the columns with wrong data type. Price and service fee is stored as text, we need to convert it to float
#construction year is a float when it should be an integer

for col in ['price', 'service fee']:
  data[col] = (data[col]
               .str.replace(r'[\$, \s]','',regex=True)
               .pipe(pd.to_numeric, errors='coerce'))


data['Construction year'] = data['Construction year'].astype('Int64')



In [10]:
#Next, lets check on category columns, like neighbourhood groups, cancellation policy, room type, etc to
#see if there are any typos in these columns.

#check typos
for col in ['neighbourhood group', 'room type', 'cancellation_policy', 'host_identity_verified', 'instant_bookable']:
    print(col, data[col].unique())

neighbourhood group ['Brooklyn' 'Manhattan' 'brookln' 'manhatan' 'Queens' nan 'Staten Island'
 'Bronx']
room type ['Private room' 'Entire home/apt' 'Shared room' 'Hotel room']
cancellation_policy ['strict' 'moderate' 'flexible' nan]
host_identity_verified ['unconfirmed' 'verified' nan]
instant_bookable [False True nan]


In [11]:
#neighbourhood group is the only one with typos, we fix that first
print(data['neighbourhood group'].value_counts(dropna=False))

data['neighbourhood group'] = data['neighbourhood group'].replace({
    'brookln': 'Brooklyn',
    'manhatan': 'Manhattan'
})

print(data['neighbourhood group'].value_counts(dropna=False)) #29 nan, we will remove them later

neighbourhood group
Manhattan        43557
Brooklyn         41630
Queens           13197
Bronx             2694
Staten Island      949
NaN                 29
manhatan             1
brookln              1
Name: count, dtype: int64
neighbourhood group
Manhattan        43558
Brooklyn         41631
Queens           13197
Bronx             2694
Staten Island      949
NaN                 29
Name: count, dtype: int64


In [12]:
# lets work on the nan category that we see in each group.
# for neighbourhood group, we will derive value from the neighbourhood column
# for other category columns - room type, cancellation policy, host identity verified, we will fill the missing value with mode (most frequent)

#recover neighbourhood group missing values
lookup = (data.dropna(subset=['neighbourhood', 'neighbourhood group'])
            .groupby('neighbourhood')['neighbourhood group']
            .agg(lambda s: s.mode()[0]))
data['neighbourhood group'] = data['neighbourhood group'].fillna(data['neighbourhood'].map(lookup))

#list out each column category with missing values to confirm mode fill up approach -> fewer than 0.3% values are missing in each column
# so filling with mode has negligible effect
for col in ['room type', 'cancellation_policy',
            'host_identity_verified', 'instant_bookable']:
    print(data[col].value_counts(dropna=False), '\n')

#mode for remaining categorical values
for col in ['neighbourhood', 'cancellation_policy', 'host_identity_verified', 'instant_bookable']:
    data[col] = data[col].fillna(data[col].mode()[0])


#verify no missing values and list the category columns
cat_cols = ['neighbourhood group', 'room type', 'cancellation_policy',
            'host_identity_verified', 'instant_bookable']
print(data[cat_cols + ['neighbourhood']].isna().sum())
for col in cat_cols:
    print(col, '->', data[col].unique())

room type
Entire home/apt    53429
Private room       46306
Shared room         2208
Hotel room           115
Name: count, dtype: int64 

cancellation_policy
moderate    34162
strict      33929
flexible    33891
NaN            76
Name: count, dtype: int64 

host_identity_verified
unconfirmed    50944
verified       50825
NaN              289
Name: count, dtype: int64 

instant_bookable
False    51186
True     50767
NaN        105
Name: count, dtype: int64 

neighbourhood group       0
room type                 0
cancellation_policy       0
host_identity_verified    0
instant_bookable          0
neighbourhood             0
dtype: int64
neighbourhood group -> ['Brooklyn' 'Manhattan' 'Queens' 'Staten Island' 'Bronx']
room type -> ['Private room' 'Entire home/apt' 'Shared room' 'Hotel room']
cancellation_policy -> ['strict' 'moderate' 'flexible']
host_identity_verified -> ['unconfirmed' 'verified']
instant_bookable -> [False  True]


/tmp/ipykernel_477/1810600442.py:19: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data[col] = data[col].fillna(data[col].mode()[0])


In [13]:
missing_data = data.isna().sum().sort_values(ascending=False)
missing_data

,0
reviews per month,15818
availability 365,448
host name,404
minimum nights,400
calculated host listings count,319
service fee,273
NAME,250
price,247
Construction year,214
number of reviews,183


In [14]:
# next we have some impossible values in minimum nights and availability 365 column, lets remove those
data.loc[(data['minimum nights'] < 1) | (data['minimum nights'] > 365), 'minimum nights'] = np.nan
data.loc[(data['availability 365'] < 0) | (data['availability 365'] > 365), 'availability 365'] = np.nan


In [15]:
# lets deal with missing values now. We dont take the approach to just drop or fill them altogether. We decide our approach based on each column.
# numeric columns - missing values get replaced with median. Mean can skew the data. Median is insensitive to outliers.
# drop columns that cant be estimated or have very little missing values
# Missing values for Price and fee can be recovered from each other


before = len(data)

# Target: no reviews means 0 reviews per month
data.loc[data['number of reviews'] == 0, 'reviews per month'] = 0

# Drop rows: location and review data can't be estimated
data = data.dropna(subset=['lat', 'long', 'number of reviews', 'reviews per month']).reset_index(drop=True)

# Price and fee: recover from each other (fee = 20% of price)
data['price'] = data['price'].fillna(data['service fee'] / 0.2)
data['service fee'] = data['service fee'].fillna(data['price'] * 0.2)

# Median for numeric features
med_cols = ['price', 'service fee', 'minimum nights', 'availability 365',
            'calculated host listings count', 'Construction year']
data[med_cols] = data[med_cols].fillna(data[med_cols].median())

# Text identifiers: not analysed, just label them
data[['NAME', 'host name']] = data[['NAME', 'host name']].fillna('Unknown')

# ===== VERIFY =====
print(f'Rows: {before} -> {len(data)} (dropped {before - len(data)})')
print('Missing values left:', data.isna().sum().sum())
print(data.shape)

Rows: 102058 -> 101845 (dropped 213)
Missing values left: 0
(101845, 20)


In [16]:
#standardize the column names
data.columns = data.columns.str.strip().str.upper().str.replace(' ', '_')
print(data.columns.tolist())

['ID', 'NAME', 'HOST_ID', 'HOST_IDENTITY_VERIFIED', 'HOST_NAME', 'NEIGHBOURHOOD_GROUP', 'NEIGHBOURHOOD', 'LAT', 'LONG', 'INSTANT_BOOKABLE', 'CANCELLATION_POLICY', 'ROOM_TYPE', 'CONSTRUCTION_YEAR', 'PRICE', 'SERVICE_FEE', 'MINIMUM_NIGHTS', 'NUMBER_OF_REVIEWS', 'REVIEWS_PER_MONTH', 'CALCULATED_HOST_LISTINGS_COUNT', 'AVAILABILITY_365']


In [17]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101845 entries, 0 to 101844
Data columns (total 20 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   ID                              101845 non-null  int64  
 1   NAME                            101845 non-null  object 
 2   HOST_ID                         101845 non-null  int64  
 3   HOST_IDENTITY_VERIFIED          101845 non-null  object 
 4   HOST_NAME                       101845 non-null  object 
 5   NEIGHBOURHOOD_GROUP             101845 non-null  object 
 6   NEIGHBOURHOOD                   101845 non-null  object 
 7   LAT                             101845 non-null  float64
 8   LONG                            101845 non-null  float64
 9   INSTANT_BOOKABLE                101845 non-null  bool   
 10  CANCELLATION_POLICY             101845 non-null  object 
 11  ROOM_TYPE                       101845 non-null  object 
 12  CONSTRUCTION_YEA

In [18]:
data.head(10)

,ID,NAME,HOST_ID,HOST_IDENTITY_VERIFIED,HOST_NAME,NEIGHBOURHOOD_GROUP,NEIGHBOURHOOD,LAT,LONG,INSTANT_BOOKABLE,CANCELLATION_POLICY,ROOM_TYPE,CONSTRUCTION_YEAR,PRICE,SERVICE_FEE,MINIMUM_NIGHTS,NUMBER_OF_REVIEWS,REVIEWS_PER_MONTH,CALCULATED_HOST_LISTINGS_COUNT,AVAILABILITY_365
0,1001254,Clean & quiet apt home by the park,80014485718,unconfirmed,Madaline,Brooklyn,Kensington,40.64749,-73.97237,False,strict,Private room,2020,966.0,193.0,10.0,9.0,0.21,6.0,286.0
1,1002102,Skylit Midtown Castle,52335172823,verified,Jenna,Manhattan,Midtown,40.75362,-73.98377,False,moderate,Entire home/apt,2007,142.0,28.0,30.0,45.0,0.38,2.0,228.0
2,1002403,THE VILLAGE OF HARLEM....NEW YORK !,78829239556,unconfirmed,Elise,Manhattan,Harlem,40.80902,-73.94190,True,flexible,Private room,2005,620.0,124.0,3.0,0.0,0.00,1.0,352.0
3,1002755,Unknown,85098326012,unconfirmed,Garry,Brooklyn,Clinton Hill,40.68514,-73.95976,True,moderate,Entire home/apt,2005,368.0,74.0,30.0,270.0,4.64,1.0,322.0
4,1003689,Entire Apt: Spacious Studio/Loft by central park,92037596077,verified,Lyndon,Manhattan,East Harlem,40.79851,-73.94399,False,moderate,Entire home/apt,2009,204.0,41.0,10.0,9.0,0.10,1.0,289.0
5,1004098,Large Cozy 1 BR Apartment In Midtown East,45498551794,verified,Michelle,Manhattan,Murray Hill,40.74767,-73.97500,True,flexible,Entire home/apt,2013,577.0,115.0,3.0,74.0,0.59,1.0,90.0
6,1004650,BlissArtsSpace!,61300605564,unconfirmed,Alberta,Brooklyn,Bedford-Stuyvesant,40.68688,-73.95596,False,moderate,Private room,2015,71.0,14.0,45.0,49.0,0.40,1.0,224.0
7,1005202,BlissArtsSpace!,90821839709,unconfirmed,Emma,Brooklyn,Bedford-Stuyvesant,40.68688,-73.95596,False,moderate,Private room,2009,1060.0,212.0,45.0,49.0,0.40,1.0,219.0
8,1005754,Large Furnished Room Near B'way,79384379533,verified,Evelyn,Manhattan,Hell's Kitchen,40.76489,-73.98493,True,strict,Private room,2005,1018.0,204.0,2.0,430.0,3.47,1.0,180.0
9,1006307,Cozy Clean Guest Room - Family Apt,75527839483,unconfirmed,Carl,Manhattan,Upper West Side,40.80178,-73.96723,False,strict,Private room,2015,291.0,58.0,2.0,118.0,0.99,1.0,90.0


In [19]:
# outlier check on our data, we kept these outlier but removed impossible values

for c in ['REVIEWS_PER_MONTH', 'MINIMUM_NIGHTS', 'CALCULATED_HOST_LISTINGS_COUNT',
          'NUMBER_OF_REVIEWS', 'PRICE']:
    q1, q3 = data[c].quantile([0.25, 0.75])
    iqr = q3 - q1
    n = ((data[c] < q1 - 1.5*iqr) | (data[c] > q3 + 1.5*iqr)).sum()
    print(f'{c}: {n} outliers ({n/len(data):.1%})')


REVIEWS_PER_MONTH: 5726 outliers (5.6%)
MINIMUM_NIGHTS: 18177 outliers (17.8%)
CALCULATED_HOST_LISTINGS_COUNT: 17681 outliers (17.4%)
NUMBER_OF_REVIEWS: 11108 outliers (10.9%)
PRICE: 0 outliers (0.0%)


In [20]:
# transformations - since reviews_per_month is our target, we calculate log(reviews_per_month)
# add a new column based on reviews_per_month
# log_reviews_per_month and high_demand is derived from reviews_per_month. To be used for charts and as ML targets.

data['LOG_REVIEWS_PER_MONTH'] = np.log1p(data['REVIEWS_PER_MONTH'])
data['HIGH_DEMAND'] = (data['REVIEWS_PER_MONTH'] >= data['REVIEWS_PER_MONTH'].median()).astype(int)
data['HOST_TYPE'] = pd.cut(data['CALCULATED_HOST_LISTINGS_COUNT'],
                         bins=[0, 1, 5, np.inf], labels=['Single', 'Small (2-5)', 'Multi (6+)'])


In [21]:
#exporting the cleaned csv to work on EDA.
data.to_csv('airbnb_cleaned.csv', index=False)
print(data.shape)

(101845, 23)


In [22]:
from google.colab import files
files.download('airbnb_cleaned.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>